# Step 05B: Ebola — Sitewise Mutation & Constraint Analysis

This notebook mirrors **05A (Lassa)** but for **Ebola**.

It uses the Step 04B outputs to:
- Characterize **sitewise conservation and mutation hotspots** in Ebola
- Identify **critical / conserved regions** vs **hotspots**
- Rank **dangerous vs harmless substitutions** (impact scores)
- Select **lab validation candidates** (rare + high-impact in conserved sites)
- Analyze **ESM outlier sequences** (unusual / adaptive variants)
- Provide helper functions to **answer mutation questions** clearly per site and substitution

Outputs are written to `results/ebola/05b/` for later comparison in 05C.


In [ ]:
# 0. Imports & Paths (RUN FIRST)

from pathlib import Path
import math
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

sns.set_theme(style="whitegrid", font_scale=1.1)

# Assume this notebook is at notebooks/Steps_Notebook
NOTEBOOK_DIR = Path(__file__).parent if "__file__" in globals() else Path(".")
REPO_ROOT = NOTEBOOK_DIR.parent.parent  # repo_root/notebooks/Steps_Notebook
print("Notebook dir:", NOTEBOOK_DIR.resolve())
print("Repo root   :", REPO_ROOT.resolve())

# --- INPUTS from Step 04B (Ebola) ---
# Adjust names if your Step 04B saved them differently
EBOLA_SITE_CSV = REPO_ROOT / "results" / "ebola" / "ebola_site_scores_reference_based.csv"
EBOLA_MUT_CSV  = REPO_ROOT / "results" / "ebola" / "ebola_mutation_scores_reference_based.csv"
EBOLA_ESM_CSV  = REPO_ROOT / "results" / "ebola" / "ebola_esm_outlier_report.csv"

# Optional (if you want direct embedding-based NN analysis)
EBOLA_EMB_PT   = REPO_ROOT / "data" / "embeddings" / "ebola_embeddings.pt"
EBOLA_META_CSV = REPO_ROOT / "data" / "embeddings" / "ebola_metadata.csv"

print("Site table :", EBOLA_SITE_CSV, "exists:", EBOLA_SITE_CSV.exists())
print("Mut table  :", EBOLA_MUT_CSV,  "exists:", EBOLA_MUT_CSV.exists())
print("ESM report :", EBOLA_ESM_CSV,  "exists:", EBOLA_ESM_CSV.exists())

# --- OUTPUTS for 05B ---
OUT_DIR = REPO_ROOT / "results" / "ebola" / "05b"
FIG_DIR = OUT_DIR / "figures"
TBL_DIR = OUT_DIR / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TBL_DIR.mkdir(parents=True, exist_ok=True)

print("Outputs will be saved under:", OUT_DIR.resolve())


In [ ]:
# 1. Load Ebola Step 04B results

if not EBOLA_SITE_CSV.exists():
    raise FileNotFoundError(f"Missing Ebola site table (Step 04B): {EBOLA_SITE_CSV}")
if not EBOLA_MUT_CSV.exists():
    raise FileNotFoundError(f"Missing Ebola mutation table (Step 04B): {EBOLA_MUT_CSV}")
if not EBOLA_ESM_CSV.exists():
    raise FileNotFoundError(f"Missing Ebola ESM outlier report (Step 04B): {EBOLA_ESM_CSV}")

site = pd.read_csv(EBOLA_SITE_CSV)
mut  = pd.read_csv(EBOLA_MUT_CSV)
esm  = pd.read_csv(EBOLA_ESM_CSV)

print("site:", site.shape)
print("mut :", mut.shape)
print("esm :", esm.shape)

display(site.head())
display(mut.head())
display(esm.head())

# Optional: load embeddings (for deeper NN/cluster analysis)
if EBOLA_EMB_PT.exists() and EBOLA_META_CSV.exists():
    emb = torch.load(EBOLA_EMB_PT, map_location="cpu")
    emb_meta = pd.read_csv(EBOLA_META_CSV)
    print("Embeddings:", type(emb), getattr(emb, "shape", None))
    print("Emb meta  :", emb_meta.shape)
else:
    emb = None
    emb_meta = None
    print("No embedding .pt or metadata CSV found for Ebola; ESM analysis limited to esm CSV.")


## 2. Global site behavior — categories, conservation, entropy

This section answers:
- How many sites are **Critical / Conserved / Hotspot / Intermediate** in Ebola?
- Where (along the reference) are the **most conserved** vs **most variable** positions?

In [ ]:
# 2.1 Category distribution
cat_counts = site["site_category"].value_counts().reset_index()
cat_counts.columns = ["site_category", "count"]
cat_counts["fraction"] = cat_counts["count"] / cat_counts["count"].sum()
display(cat_counts)

cat_counts.to_csv(TBL_DIR / "ebola_site_category_summary.csv", index=False)

if "ref_pos" in site.columns:
    print("Max ref_pos:", site["ref_pos"].max())
else:
    print("WARNING: site table missing ref_pos column")


In [ ]:
# 2.2 Conservation and entropy curves

plt.figure(figsize=(16,4))
plt.plot(site["ref_pos"], site["conservation"], linewidth=1)
plt.xlabel("Reference position")
plt.ylabel("Conservation (0–1)")
plt.title("Ebola: conservation by reference position")
plt.tight_layout()
plt.savefig(FIG_DIR / "ebola_conservation_curve.png", dpi=300)
plt.show()

plt.figure(figsize=(16,4))
plt.plot(site["ref_pos"], site["entropy"], linewidth=1, color="darkorange")
plt.xlabel("Reference position")
plt.ylabel("Entropy")
plt.title("Ebola: entropy (variability) by reference position")
plt.tight_layout()
plt.savefig(FIG_DIR / "ebola_entropy_curve.png", dpi=300)
plt.show()


In [ ]:
# 2.3 Category barcode track

plt.figure(figsize=(16,2))
color_map = dict(
    Critical="red",
    Conserved="forestgreen",
    Hotspot="gold",
    Intermediate="gray",
    MostlyGap="lightgray",
)
colors = [color_map.get(c, "black") for c in site["site_category"]]
plt.scatter(site["ref_pos"], [1]*len(site), c=colors, marker="|", s=20)
plt.yticks([])
plt.xlabel("Reference position")
plt.title("Ebola: site category track")
plt.tight_layout()
plt.savefig(FIG_DIR / "ebola_site_category_barcode.png", dpi=300)
plt.show()


## 3. Hotspots and critical sites

This section finds:
- **Hotspots**: high-entropy "Hotspot" sites (most variable)
- **Critical**: highly conserved sites that likely cannot tolerate mutation

In [ ]:
# 3.1 Hotspots and critical sites tables

hotspots = site[site["site_category"] == "Hotspot"].sort_values("entropy", ascending=False)
critical = site[site["site_category"] == "Critical"].sort_values("conservation", ascending=False)

hotspots.head(200).to_csv(TBL_DIR / "ebola_top_hotspots.csv", index=False)
critical.head(200).to_csv(TBL_DIR / "ebola_top_critical_sites.csv", index=False)

print("Hotspots:", hotspots.shape[0])
print("Critical:", critical.shape[0])

display(hotspots.head(10))
display(critical.head(10))


## 4. Substitution-level risk: dangerous vs harmless mutations

Here we analyze the substitution scores from Step 04B:
- Global distribution of **impact_score**
- Top dangerous and top harmless substitutions
- Basis for lab validation and future model explanations (Step 06).

In [ ]:
# 4.1 Distribution of impact scores

plt.figure(figsize=(10,4))
sns.histplot(mut["impact_score"], bins=40, kde=True)
plt.xlabel("Impact score (0–100)")
plt.title("Ebola: distribution of substitution impact scores")
plt.tight_layout()
plt.savefig(FIG_DIR / "ebola_mutation_impact_distribution.png", dpi=300)
plt.show()

# 4.2 Top dangerous vs harmless substitutions

top_danger = mut.sort_values("impact_score", ascending=False)
top_harmless = mut.sort_values("impact_score", ascending=True)

top_danger.head(500).to_csv(TBL_DIR / "ebola_top_dangerous_mutations.csv", index=False)
top_harmless.head(500).to_csv(TBL_DIR / "ebola_top_harmless_mutations.csv", index=False)

display(top_danger.head(10))
display(top_harmless.head(10))


## 5. Lab validation candidates

High-priority candidates are:
- At **Critical/Conserved** sites
- Have **rare** alt amino acid (low frequency in dataset)
- Have **high impact_score**

In [ ]:
# 5.1 Rare + high-impact at conserved/critical sites

lab = mut[
    (mut["site_category"].isin(["Critical", "Conserved"])) &
    (mut["alt_freq_in_dataset"] <= 0.001) &
    (mut["impact_score"] >= 80)
].sort_values(["impact_score", "site_conservation"], ascending=[False, False])

lab.head(500).to_csv(TBL_DIR / "ebola_lab_validation_candidates.csv", index=False)
print("Lab candidates:", lab.shape[0])
display(lab.head(15))


## 6. ESM outliers: unusual Ebola sequences

This section uses the Step 04B ESM outlier report to find:
- Top outlier sequences (unusual embeddings)
- Distribution of outlier scores
- Optionally, relation with sequence length (QC / adaptation hints).

In [ ]:
# 6.1 Top outlier sequences

assert "esm_outlier_score" in esm.columns, "ESM outlier score column missing in Ebola ESM CSV."

outliers = esm.sort_values("esm_outlier_score", ascending=False)
outliers.head(200).to_csv(TBL_DIR / "ebola_outlier_sequences.csv", index=False)
display(outliers.head(10))

# 6.2 Outlier distribution

plt.figure(figsize=(10,4))
sns.histplot(esm["esm_outlier_score"], bins=35, kde=True)
plt.xlabel("ESM outlier score (0–100)")
plt.title("Ebola: ESM outlier score distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "ebola_outlier_score_distribution.png", dpi=300)
plt.show()

# 6.3 Outlier vs length (if length column exists)

if "length" in esm.columns:
    plt.figure(figsize=(8,5))
    sns.scatterplot(x=esm["length"], y=esm["esm_outlier_score"], alpha=0.7)
    plt.xlabel("Sequence length")
    plt.ylabel("ESM outlier score")
    plt.title("Ebola: outlier score vs length")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "ebola_outlier_vs_length.png", dpi=300)
    plt.show()
else:
    print("No 'length' column in Ebola ESM CSV; skipping outlier vs length plot.")


## 7. Helper to answer mutation questions at a site

This function lets you query: given `(ref_pos, ref_aa, alt_aa)`, return the full record from `ebola_mutation_scores_reference_based.csv`.

This is the key building block for Step 06, where a user inputs mutations and the model explains: 
- which site,
- what category (Critical / Hotspot),
- how risky (impact_score, impact_category),
- and how common that alt amino acid is.

In [ ]:
def query_ebola_mutation(ref_pos: int, ref_aa: str, alt_aa: str):
    q = mut[
        (mut["ref_pos"] == ref_pos) &
        (mut["ref_aa"] == ref_aa) &
        (mut["alt_aa"] == alt_aa)
    ]
    if q.empty:
        return None
    return q.iloc[0].to_dict()

# Example (adjust to a real position and AAs from your dataset):
# example = query_ebola_mutation(123, "A", "D")
# example


### Optional: score a list of user mutations for Ebola

This small helper is for Step 06; here we just provide it to keep 05B complete.

Given a list like `[{"pos": 123, "ref": "A", "alt": "D"}, ...]` it looks up each in the Ebola mutation table and returns a DataFrame with impact and category if available.

In [ ]:
def score_ebola_mutation_list(mutation_list):
    rows = []
    for m in mutation_list:
        pos = m["pos"]
        ref = m["ref"]
        alt = m["alt"]
        rec = query_ebola_mutation(pos, ref, alt)
        if rec is None:
            rows.append({**m, "impact_score": None, "impact_category": None, "note": "not found in table"})
        else:
            rows.append({**m, **rec})
    return pd.DataFrame(rows)

# Example usage (uncomment when you have real positions):
# user_muts = [
#     {"pos": 123, "ref": "A", "alt": "D"},
#     {"pos": 456, "ref": "G", "alt": "V"},
# ]
# score_ebola_mutation_list(user_muts)


## 8. (Optional) Embedding-based nearest neighbors (if raw embeddings loaded)

If you loaded `ebola_embeddings.pt` and `ebola_metadata.csv`, you can:
- compute centroid-based outlier scores (independent sanity check),
- get nearest neighbors for a given Ebola sequence in embedding space.

This is not required for 05B, but useful for deeper sequence-level insight or QC.

In [ ]:
def cosine_distance_to_centroid(X: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    Xn = X / (torch.linalg.norm(X, dim=1, keepdim=True) + eps)
    c = Xn.mean(dim=0)
    sim = Xn @ c
    return 1.0 - sim

if emb is not None and emb_meta is not None and isinstance(emb, torch.Tensor):
    X = emb.float()
    d = cosine_distance_to_centroid(X)
    # simple robust z-score
    med = d.median()
    mad = (d - med).abs().median()
    z = (d - med) / (1.4826 * mad + 1e-12)
    # map to 0–100
    zc = torch.clamp(z, -6, 6)
    score = (zc + 6) * (100.0 / 12.0)
    emb_meta["esm_outlier_from_embedding"] = score.tolist()
    emb_meta.sort_values("esm_outlier_from_embedding", ascending=False).head(100).to_csv(
        TBL_DIR / "ebola_embedding_based_top_outliers.csv", index=False
    )
    print("Saved embedding-based top outliers to:", TBL_DIR / "ebola_embedding_based_top_outliers.csv")
else:
    print("Embedding tensors/metadata not available or invalid, skipping deeper NN/outlier analysis.")


## 9. Summary of outputs

This cell prints the main tables and figures generated in 05B.

In [ ]:
print("05B Ebola analysis complete.")
print("Tables saved in:", TBL_DIR)
print("Figures saved in:", FIG_DIR)

print("\nTables:")
for p in sorted(TBL_DIR.glob("*.csv")):
    print(" -", p.name)

print("\nFigures:")
for p in sorted(FIG_DIR.glob("*.png")):
    print(" -", p.name)
